This code simulates a user's transaction history and engineers features that flag
- device tampering (changing language/OS suddenly) and
- device sharing (too many accounts using one device).

In [2]:
import pandas as pd

# 1. Create mock transaction history with device details
data = {
    "tx_id": [1, 2, 3, 4, 5, 6],
    "user_id": ["user_A", "user_A", "user_A", "user_B", "user_C", "user_D"],
    "timestamp": pd.to_datetime([
        "2026-05-18 10:00:00", 
        "2026-05-18 10:05:00", 
        "2026-05-18 10:10:00",
        "2026-05-18 11:00:00",
        "2026-05-18 11:01:00",
        "2026-05-18 11:02:00"
    ]),
    # Combined device features acting as a unique signature (fingerprint)
    "device_hash": ["dev_999", "dev_999", "dev_777", "dev_444", "dev_444", "dev_444"],
    "device_os": ["iOS", "iOS", "Android", "Windows", "Windows", "Windows"],
    "device_lang": ["en-US", "en-US", "ru-RU", "en-US", "en-US", "en-US"]
}
df = pd.DataFrame(data).sort_values(["user_id", "timestamp"])

df

,tx_id,user_id,timestamp,device_hash,device_os,device_lang
0,1,user_A,2026-05-18 10:00:00,dev_999,iOS,en-US
1,2,user_A,2026-05-18 10:05:00,dev_999,iOS,en-US
2,3,user_A,2026-05-18 10:10:00,dev_777,Android,ru-RU
3,4,user_B,2026-05-18 11:00:00,dev_444,Windows,en-US
4,5,user_C,2026-05-18 11:01:00,dev_444,Windows,en-US
5,6,user_D,2026-05-18 11:02:00,dev_444,Windows,en-US


In [3]:
# --- FEATURE ENGINEERING ---

# FEATURE 1: Device Configuration Fluctuation (Per User)
# Tracks if the user's OS or language suddenly changes from their last transaction
df["prev_os"] = df.groupby("user_id")["device_os"].shift(1)
df["prev_lang"] = df.groupby("user_id")["device_lang"].shift(1)

df["device_os_changed"] = (df["device_os"] != df["prev_os"]) & df["prev_os"].notna()
df["device_lang_changed"] = (df["device_lang"] != df["prev_lang"]) & df["prev_lang"].notna()
df["device_tampered_flag"] = df["device_os_changed"] | df["device_lang_changed"]

# FEATURE 2: Device Hijacking / Account Multiplicity (Per Device)
# Counts how many unique users have used this exact device in the last 24 hours
# (Simulating a fraud ring using a single laptop/phone to open dozens of fake profiles)
df["unique_users_per_device"] = (
    df.groupby("device_hash")["user_id"]
    .transform("nunique")
)

# Flag if a single device is shared by 3 or more distinct user accounts
df["shared_device_fraud_flag"] = df["unique_users_per_device"] >= 3

# Display engineered features
print(df[[
    "tx_id", 
    "user_id",
    "device_hash", 
    "device_tampered_flag", 
    "unique_users_per_device", 
    "shared_device_fraud_flag"
]])


   tx_id user_id device_hash  device_tampered_flag  unique_users_per_device  \
0      1  user_A     dev_999                 False                        1   
1      2  user_A     dev_999                 False                        1   
2      3  user_A     dev_777                  True                        1   
3      4  user_B     dev_444                 False                        3   
4      5  user_C     dev_444                 False                        3   
5      6  user_D     dev_444                 False                        3   

   shared_device_fraud_flag  
0                     False  
1                     False  
2                     False  
3                      True  
4                      True  
5                      True  


## Transaction 3 (device_tampered_flag = True): 
- user_A suddenly switched from an iPhone (iOS/en-US) to a Russian Android phone (Android/ru-RU) **within 5 minutes**. 
- This triggers an immediate alert for account takeover or device spoofing.

## Transactions 4, 5, & 6 (shared_device_fraud_flag = True): 
- Three completely different users (user_B, user_C, user_D) checked out back-to-back using the exact **same hardware fingerprint** (dev_444). 
- This is a classic pattern for a fraud farm or promo-abuse botnet.